In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# =========================
# LOAD DATA
# =========================

df = pd.read_csv("/kaggle/input/datasets/souptik22fiitbs/numerical-data/blood_respiratory_dataset_plus_1900_outliers_8900.csv")

FEATURE_COLS = [
    "WBC (x10^9/L)",
    "NEUT%",
    "LYMP%",
    "NLR",
    "CRP (mg/L)",
    "PCT (ng/mL)"
]

X = df[FEATURE_COLS].values
y = df["Diagnosis_Label"]

# =========================
# LABEL ENCODING
# =========================

le = LabelEncoder()
y = le.fit_transform(y)

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# STANDARDIZATION
# =========================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# =========================
# BASELINE MODEL
# =========================

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

# =========================
# PREDICTION
# =========================

y_pred = model.predict(X_test)

# =========================
# METRICS
# =========================

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')

# =========================
# SPECIFICITY
# =========================

cm = confusion_matrix(y_test, y_pred)

specificities = []

for i in range(len(cm)):
    TP = cm[i, i]
    FN = np.sum(cm[i, :]) - TP
    FP = np.sum(cm[:, i]) - TP
    TN = np.sum(cm) - (TP + FN + FP)

    specificity = TN / (TN + FP)
    specificities.append(specificity)

overall_specificity = np.mean(specificities)

# =========================
# RESULTS
# =========================

print(f"\nAccuracy     : {accuracy:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"Specificity  : {overall_specificity:.4f}\n")

print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))


Accuracy     : 0.8045
Precision    : 0.8098
Recall       : 0.8045
F1 Score     : 0.8038
Specificity  : 0.9511

                 precision    recall  f1-score   support

      Bacterial       0.99      0.91      0.95       356
       COVID-19       0.71      0.78      0.74       356
        Healthy       0.80      0.96      0.88       356
   Tuberculosis       0.70      0.67      0.68       356
Viral Pneumonia       0.85      0.71      0.77       356

       accuracy                           0.80      1780
      macro avg       0.81      0.80      0.80      1780
   weighted avg       0.81      0.80      0.80      1780

